# GISGPT — ดึงข้อมูลจาก GISTDA Open Data (CKAN API)

GISTDA Open Data (https://opendata.gistda.or.th) เป็นระบบ CKAN — ข้อมูลไม่ได้เก็บใน GitHub แต่เข้าถึงผ่าน API สด

**API 2 แบบ:**
1. **CKAN API** (เปิดใช้งานได้ทันที ไม่ต้องใช้คีย์) — ค้นหา/ดูรายการชุดข้อมูล และดึง URL ของ service
2. **API Gateway** (ต้องสมัครคีย์ฟรีที่ https://api-gateway.gistda.or.th) — ดึงข้อมูลจริง เช่น พื้นที่ปลูกข้าว, น้ำท่วม

In [ ]:
import json
import requests

CKAN = 'https://opendata.gistda.or.th/api/3/action/'
print('ready')

## 1. ค้นหาชุดข้อมูลด้วย CKAN API (ไม่ต้องใช้คีย์)

`package_search` คล้ายการค้นหาบนเว็บ เช่น ค้นหาคำว่า "น้ำท่วม" หรือ "ข้าว"

In [ ]:
def search_datasets(query, limit=10):
    r = requests.get(CKAN + 'package_search',
                     params={'q': query, 'rows': limit}, timeout=30)
    r.raise_for_status()
    results = r.json()['result']['results']
    print(f'เจอทั้งหมด {r.json()["result"]["count"]} ชุด แสดง {len(results)} ชุดแรก:')
    for d in results:
        print(f"  - {d['name']:<28} | {d.get('title', '')[:40]}")
    return results

search_datasets('น้ำท่วม')

## 2. ดูรายละเอียดชุดข้อมูล (package_show)

ดึง resource ทั้งหมดของชุดข้อมูล เพื่อดู URL ของ service ที่เราจะเรียกใช้

In [ ]:
def show_dataset(dataset_id):
    r = requests.get(CKAN + 'package_show', params={'id': dataset_id}, timeout=30)
    r.raise_for_status()
    d = r.json()['result']
    print('ชื่อชุดข้อมูล:', d['title'])
    for res in d['resources']:
        print(f'  resource: {res["name"][:45]:<47} format: {res["format"]}')
        print(f'    url: {res["url"][:90]}')
    return d

show_dataset('disasters-01')

## 3. เตรียม API key สำหรับ API Gateway

- สมัครฟรีที่ https://api-gateway.gistda.or.th (ลงทะเบียนด้วยอีเมล)
- กรอกคีย์ด้านล่าง (คีย์ใน URL สาธารณะของ GISTDA **หมดอายุแล้ว** ต้องใช้ของตัวเอง)
- ถ้ายังไม่มีคีย์ รันข้ามไปได้เลย — notebook จะแสดงวิธีใช้ให้

In [ ]:
GATEWAY = 'https://api-gateway.gistda.or.th/api/2.0/'
API_KEY = ''  # ใส่คีย์ของคุณ เช่น API_KEY = 'xxxxxxxxxxxxxxxxxxxxxxxxxxxx'

if not API_KEY:
    print('ยังไม่ใส่ API key → ข้ามการเรียกข้อมูลจริง (ดูตัวอย่าง endpoint ด้านล่าง)')
else:
    print('API key พร้อมใช้งาน')

## 4. เรียกข้อมูลจริง: พื้นที่ปลูกข้าวรายสัปดาห์ (rice-weekly-40m)

Endpoint จากชุดข้อมูล `dataset_2024_03` (พื้นที่ปลูกข้าวของประเทศไทย)

แบบ **จุด** (lat/lon) — ถามว่าตำแหน่งนี้กำลังปลูกข้าวหรือไม่

In [ ]:
if API_KEY:
    url = (GATEWAY + 'resources/gi-service/v2.2/agriculture/rice-weekly-40m'
           + '?lat=14.129957&lon=100.259461&api_key=' + API_KEY)
    resp = requests.get(url, timeout=30)
    if resp.status_code == 200:
        print('ผลลัพธ์:')
        print(json.dumps(resp.json(), ensure_ascii=False, indent=2)[:800])
    else:
        print(f'Error {resp.status_code}: ตรวจสอบ API key หรือสิทธิ์ใช้งาน')
else:
    print('ตัวอย่าง endpoint (ใส่ API_KEY แล้วรันใหม่):')
    print('https://api-gateway.gistda.or.th/api/2.0/resources/gi-service/v2.2/agriculture/rice-weekly-40m?lat=14.129957&lon=100.259461&api_key=YOUR_KEY')

## 5. สรุปแนวทางใช้ใน GISGPT

1. ผู้ใช้เลือก ROI บนแผนที่ใน prototype → เอา bounds/polygon ไป query API Gateway
2. ใช้ CKAN API ค้นหาชุดข้อมูลที่เกี่ยวข้องกับคำถามของผู้ใช้
3. ภาพดาวเทียม 2 เมตร (gi-basemap) ใช้เป็นเลเยอร์แผนที่ได้ — ต้องใส่ API key ใน `prototype/js/app.js` (ตัวแปร `GISTDA_KEY`)

ชุดข้อมูลที่น่าสนใจ: `dataset_2024_03` (ข้าว), `dataset_2024_01` (อ้อย), `disasters-01` (น้ำท่วมซ้ำซาก), `disasters-04` (พื้นที่เผาไหม้), `pm2-5` (ฝุ่น)